# Completed SPY Merton / LSTM project
Run cells in order. This notebook uses the packaged Python source, so its formulas and splits cannot drift from the CLI.

The historical calibration task and the next-21-session volatility forecasting task are separate. Results can be poor even when the code is correct. The supplied test set was previously examined; it is kept out of training here but is not a fresh scientific holdout.


In [ ]:
from pathlib import Path
import sys, os, zipfile, importlib.util, tempfile
os.environ.setdefault("PIP_CACHE_DIR", str(Path(tempfile.gettempdir()) / "finance-pip-cache"))
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()  # Upload the COMPLETE project ZIP.
    names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(names) != 1:
        raise ValueError("Upload exactly one complete project ZIP.")
    destination = Path("/content/completed_project").resolve()
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(names[0]) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError("Unsafe ZIP member path.")
        archive.extractall(destination)
    roots = [p.parent for p in destination.rglob("run_project.py") if (p.parent / "config.json").exists()]
else:
    roots = [p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_project.py").exists()]
if len(roots) != 1:
    raise RuntimeError("Cannot identify the project. Start this notebook from the extracted project folder.")
ROOT = roots[0].resolve()
os.chdir(ROOT)
print("Project located. Raw data and source are available.")


TypeError: 'NoneType' object is not subscriptable

In [ ]:
import subprocess
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.14.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
else:
    # Install requirements into this notebook's Python environment first (README).
    subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
print("Using this notebook's Python environment.")


## Run from raw data
This recalibrates all historical windows, trains both model tasks, selects on validation, freezes choices and finally evaluates test dates. Forward labels crossing split boundaries are purged. No cached model or fit is reused.


In [ ]:
subprocess.run([sys.executable, "run_project.py"], check=True)


## Test and independently verify
Tests include mathematical identities, Poisson normalization, analytic gradients, parameter recovery on simulated data, date alignment, leakage, deterministic model reloads and a small end-to-end run. Verification recomputes forecasts and metrics from saved artifacts.


In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
subprocess.run([sys.executable, "src/verify_project.py"], check=True)


In [ ]:
import pandas as pd
from IPython.display import display, Markdown, Image
metrics = pd.read_csv("results/metrics/future_volatility.csv")
display(metrics[metrics["split"] == "test"])
display(pd.read_csv("results/metrics/parameter_metrics.csv"))
display(Markdown(Path("report/project_summary.md").read_text()))
for name in ["training", "parameters", "future_volatility", "errors", "calibration_diagnostics"]:
    display(Image(filename=f"results/figures/{name}.png"))


## Reload and predict
The saved example includes all historical returns needed to reconstruct sequence, summary and EWMA features. It uses no future prices.


In [ ]:
sys.path.insert(0, str(ROOT / "src"))
from finance_project.pipeline import example_predict
print(example_predict(ROOT))


## Save the completed run
In Colab this downloads a backup ZIP. Locally the folder already contains every result. Preserve the complete folder, including model preprocessing files.


In [ ]:
if IN_COLAB:
    backup = ROOT.parent / "finance_project_completed_backup.zip"
    with zipfile.ZipFile(backup, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in ROOT.rglob("*"):
            if path.is_file() and not any(part in {".venv", "__pycache__", ".pytest_cache", ".ipynb_checkpoints"} for part in path.relative_to(ROOT).parts):
                archive.write(path, Path(ROOT.name) / path.relative_to(ROOT))
    files.download(str(backup))
else:
    print("Completed results are saved in this project folder.")
